In [ ]:
#%%writefile C:\Users\neele\Music\Travscape\agents\planner.py
from agents.utils import get_today_str
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from agents.GlobalState import AgentState
from agents.planner_output import PlannerOutput
from agents.prompts import planner_message
from agents.utils import get_today_str

load_dotenv(override=True)

################Example:
trip_details = {
      "destination": "Tokyo",
      "dates": "October 10, 2025",
      "no._of_days": 5,
      "travelers": 3,
      "budget": "Budget trip",
      "purpose": "sightseeing",
      "preferences": None
    }
################

planner = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

planner_with_output = planner.with_structured_output(PlannerOutput)

#System message initialization with today's date
system_message = planner_message.format(
  date=get_today_str(),
  trip_details=trip_details
)

def Planner_node(state:AgentState) -> AgentState:
  
    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True

    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages

    try:
        response = planner_with_output.invoke(messages)

        if response.need_clarification and response.clarification_question is not None:
          ai_response = AIMessage(content=response.clarification_question)
          updated_messages = messages + [ai_response]

          return Command(
              update={
                  "messages": updated_messages
              }
          )

        else:
            plan_dict = response.plan.model_dump() if response.plan else {}
            ai_response = AIMessage(content=f"Plan generated: {plan_dict}")
            updated_messages = messages + [ai_response]

            return Command(
                update={
                    "messages": updated_messages,
                    "trip_plan": [plan_dict]
                }
            )
        
    except Exception as e:
        print(f"Error: {e}")
        error_message = AIMessage(content="Sorry, I encountered an error. Please try again.")
        return Command(
            update={"messages": messages + [error_message]}
        )
    
checkpointer = InMemorySaver()

planner_builder = StateGraph(AgentState)

planner_builder.add_node("Planner", Planner_node)

planner_builder.add_edge(START, "Planner")
planner_builder.add_edge("Planner", END)

graph = planner_builder.compile(checkpointer=checkpointer)



In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
import gradio as gr
config = {"configurable": {"thread_id": "1"}}

def chat(message, history):
    try:
        # Extract user message content
        user_message = message if isinstance(message, str) else message.get("text", "")
        
        # Invoke the graph
        result = graph.invoke(
            {"messages": [HumanMessage(content=user_message)]}, 
            config=config
        )
        
        # Return the assistant's response in proper format
        assistant_response = result["messages"][-1].content
        return assistant_response
        
    except Exception as e:
        print(f"Chat error: {e}")
        return (f"Sorry, I encountered an error. -> {e}")

gr.ChatInterface(
    chat, 
    type="messages",
    title="Trav - Your Travel Planning Assistant",
    description="Hi! I'm Trav, ready to help you plan your next adventure!"
).launch()

In [ ]:
#%%writefile C:\Users\neele\Music\Travscape\agents\orchestrator.py

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.types import Command
from agents.GlobalState import AgentState
from agents.orchestrator_output import OrchestratorDecision
from agents.prompts import orchestrator_message
from agents.utils import get_today_str
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv(override=True)

In [1]:
import httpx
import os
from langchain.tools import tool
from typing import Optional, Dict, Any
from dotenv import load_dotenv

load_dotenv(override=True)

def search_flights(
    departure_id: str,
    arrival_id: str,
    outbound_date: str,
    return_date: str = "",
    travel_class: str = "ECONOMY",
    adults: str = "1",
    children: str = "0",
    infants: str = "0",
    show_hidden: str = "1",
    currency: str = "USD",
    language_code: str = "en-us",
    country_code: str = "US",
    search_type: str = "best",
) -> Dict[str, Any]:

    url = "https://google-flights2.p.rapidapi.com/api/v1/searchFlights"

    query = {
        "departure_id": departure_id,
        "arrival_id": arrival_id,
        "outbound_date": outbound_date,
        "travel_class": travel_class,
        "adults": adults,
        "children": children,
        "infant_on_lap": infants,
        "show_hidden": show_hidden,
        "currency": currency,
        "language_code": language_code,
        "country_code": country_code,
        "search_type": search_type,
    }

    # Only include return_date if provided
    if return_date and return_date.strip():
        query["return_date"] = return_date

    headers = {
        "x-rapidapi-key": "7f0827a6b8mshe26589c7fa2c1a1p17490cjsne3f1eecb0923",
        "x-rapidapi-host": "google-flights2.p.rapidapi.com",
    }

    try:
        resp = httpx.get(url, headers=headers, params=query, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        return {"error": str(e)}

    payload = resp.json()

    itineraries = payload.get("data", {}).get("itineraries", {}) or {}
    top_flights = itineraries.get("topFlights", []) or []
    other_flights = itineraries.get("otherFlights", []) or []

    # ---- Minimal LEG extractor ----
    def build_leg(leg: Dict[str, Any]) -> Dict[str, Any]:
        dep = leg.get("departure_airport") or {}
        arr = leg.get("arrival_airport") or {}
        dur = leg.get("duration") or {}

        return {
            "departure_airport_code": dep.get("airport_code"),
            "departure_airport_name": dep.get("airport_name"),
            "departure_time": dep.get("time"),
            "arrival_airport_code": arr.get("airport_code"),
            "arrival_airport_name": arr.get("airport_name"),
            "arrival_time": arr.get("time"),
            "leg_duration_text": dur.get("text"),
            "airline": leg.get("airline"),
            "airline_logo": leg.get("airline_logo"),
            "flight_number": leg.get("flight_number"),
        }

    # ---- Minimal Itinerary extractor ----
    def normalize_itinerary(itin: Dict[str, Any]) -> Dict[str, Any]:
        raw_flights = itin.get("flights")
        if raw_flights is None:
            flights_list = []
        elif isinstance(raw_flights, dict):
            flights_list = [raw_flights]
        elif isinstance(raw_flights, list):
            flights_list = raw_flights
        else:
            flights_list = []

        flights_min = [build_leg(f) for f in flights_list]

        return {
            "departure_time": itin.get("departure_time"),
            "arrival_time": itin.get("arrival_time"),
            "duration_text": (itin.get("duration") or {}).get("text"),
            "price": itin.get("price"),
            "stops": itin.get("stops"),
            "booking_token": itin.get("booking_token"),
            "flights": flights_min,
        }

    normalized_top = [normalize_itinerary(it) for it in top_flights]
    #normalized_other = [normalize_itinerary(it) for it in other_flights]

    return {
        "top_itineraries": normalized_top,
        #"other_itineraries": normalized_other,
    }

result = search_flights(
    departure_id="DEL",
    arrival_id="JFK",
    outbound_date="2025-11-25",
    return_date="2025-12-05",
    travel_class="ECONOMY",
    adults="1",
    children="0",
    infants="0",
    show_hidden="1",
    currency="USD",
    language_code="en",
    country_code="US",
    search_type="best"
)
print(result)

{'top_itineraries': []}


In [4]:
import httpx
import os
from langchain.tools import tool
from typing import Optional, Dict, Any
from dotenv import load_dotenv

load_dotenv(override=True)

#Get City Code
def get_airport_code(location):
  """This function searches the source's and destination's airport ID for a given location using the Booking.com API. The first step when searching for flights."""

  url = "https://booking-com15.p.rapidapi.com/api/v1/flights/searchDestination"

  querystring = {"query":location}

  headers = {
    "x-rapidapi-key": os.getenv('x-rapidapi-key'),
    "x-rapidapi-host": "booking-com15.p.rapidapi.com"
  }
  try:
    response = httpx.get(url, headers=headers, params=querystring)
    response.raise_for_status()  # Raise an error for bad responses
    airport_code = response.json().get('data', [])
    
    if not airport_code:
        print("No destinations found for this query.")
        airport_details = []

    airport_details = [dest["code"] for dest in airport_code] #Saves all airport IDs for the given location
  
  except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
  except httpx.RequestError as e:
    print(f"Request error occurred: {e}") 
  
  return airport_details

def search_flights(
    departure_id: str,
    arrival_id: str,
    outbound_date: str,
    return_date: str,
    travel_class: str,
    adults: str,
    children: str,
    infants: str,
    show_hidden: str,
    currency: str,
    language_code: str,
    country_code: str,
    search_type: str,
) -> Dict[str, Any]:

    url = "https://google-flights2.p.rapidapi.com/api/v1/searchFlights"

    query = {
        "departure_id": departure_id,
        "arrival_id": arrival_id,
        "outbound_date": outbound_date,
        "travel_class": travel_class,
        "adults": adults,
        "children": children,
        "infant_on_lap": infants,
        "show_hidden": show_hidden,
        "currency": currency,
        "language_code": language_code,
        "country_code": country_code,
        "search_type": search_type,
    }

    # Only include return_date if provided
    if return_date and return_date.strip():
        query["return_date"] = return_date

    headers = {
        "x-rapidapi-key": os.getenv('x-rapidapi-key'),
        "x-rapidapi-host": "google-flights2.p.rapidapi.com",
    }

    try:
        resp = httpx.get(url, headers=headers, params=query, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        return {"error": str(e)}

    payload = resp.json()

    itineraries = payload.get("data", {}).get("itineraries", {}) or {}
    top_flights = itineraries.get("topFlights", []) or []
    other_flights = itineraries.get("otherFlights", []) or []

    # ---- Minimal LEG extractor ----
    def build_leg(leg: Dict[str, Any]) -> Dict[str, Any]:
        dep = leg.get("departure_airport") or {}
        arr = leg.get("arrival_airport") or {}
        dur = leg.get("duration") or {}

        return {
            "departure_airport_code": dep.get("airport_code"),
            "departure_airport_name": dep.get("airport_name"),
            "departure_time": dep.get("time"),
            "arrival_airport_code": arr.get("airport_code"),
            "arrival_airport_name": arr.get("airport_name"),
            "arrival_time": arr.get("time"),
            "leg_duration_text": dur.get("text"),
            "airline": leg.get("airline"),
            "airline_logo": leg.get("airline_logo"),
            "flight_number": leg.get("flight_number"),
        }

    # ---- Minimal Itinerary extractor ----
    def normalize_itinerary(itin: Dict[str, Any]) -> Dict[str, Any]:
        raw_flights = itin.get("flights")
        if raw_flights is None:
            flights_list = []
        elif isinstance(raw_flights, dict):
            flights_list = [raw_flights]
        elif isinstance(raw_flights, list):
            flights_list = raw_flights
        else:
            flights_list = []

        flights_min = [build_leg(f) for f in flights_list]

        return {
            "departure_time": itin.get("departure_time"),
            "arrival_time": itin.get("arrival_time"),
            "duration_text": (itin.get("duration") or {}).get("text"),
            "price": itin.get("price"),
            "stops": itin.get("stops"),
            "booking_token": itin.get("booking_token"),
            "flights": flights_min,
        }

    normalized_top = [normalize_itinerary(it) for it in top_flights]
    #normalized_other = [normalize_itinerary(it) for it in other_flights]

    return {
        "top_itineraries": normalized_top,
        #"other_itineraries": normalized_other,
    }

#@tool
def flight_search_tool(
    departure: str,
    arrival: str,
    outbound_date: str,
    return_date: str = "",
    travel_class: str = "ECONOMY",
    adults: str = "1",
    children: str = "0",
    infants: str = "0",
    show_hidden: str = "1",
    currency: str = "INR",
    language_code: str = "en-US",
    country_code: str = "IN",
    search_type: str = "best",
) -> Dict[str, Any]:
    
    """Combined tool to search for flights using the provided parameters."""
    departure_code = get_airport_code(departure)
    arrival_code = get_airport_code(arrival)
    print(departure_code, arrival_code)

    # if not departure_code or arrival_code:
    #     return {"error": "Could not find airport codes for the provided locations."}
    
    result= search_flights(
        departure_id=departure_code[1],
        arrival_id=arrival_code[1],
        outbound_date=outbound_date,
        return_date=return_date,
        travel_class=travel_class,
        adults=adults,
        children=children,
        infants=infants,
        show_hidden=show_hidden,
        currency=currency,
        language_code=language_code,
        country_code=country_code,
        search_type=search_type,
    )

    return result

In [5]:
result = flight_search_tool(departure="New Delhi",arrival="New York",outbound_date="2025-11-25",travel_class="ECONOMY",adults="1",currency="INR")
print(result)

['DEL'] ['NYC', 'JFK', 'EWR', 'LGA', 'BUF', 'ALB', 'ISP', 'SYR', 'HPN', 'ROC', 'HTS']


IndexError: list index out of range